In [2]:
import shutil

# Kreiraj direktorij ako ne postoji
!mkdir -p ~/.kaggle

# Premjesti kaggle.json u ~/.kaggle/
shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")

# Postavi odgovarajuće dozvole
!chmod 600 ~/.kaggle/kaggle.json


In [3]:
!kaggle datasets download -d balraj98/deepglobe-land-cover-classification-dataset -p /content/data --unzip


Dataset URL: https://www.kaggle.com/datasets/balraj98/deepglobe-land-cover-classification-dataset
License(s): other


In [4]:
import os

dataset_path = "/content/data"  # Postavi točnu putanju do dataseta
for root, dirs, files in os.walk(dataset_path):
    print(root)
    for file in files[:5]:  # Prikaz prvih 5 datoteka po direktoriju
        print(f"  {file}")


/content/data
  class_dict.csv
  metadata.csv
/content/data/valid
  609620_sat.jpg
  290582_sat.jpg
  456386_sat.jpg
  541127_sat.jpg
  706496_sat.jpg
/content/data/train
  935193_sat.jpg
  717225_mask.png
  246378_mask.png
  775304_sat.jpg
  899693_mask.png
/content/data/test
  780313_sat.jpg
  289091_sat.jpg
  6520_sat.jpg
  958951_sat.jpg
  805304_sat.jpg


Priprema labela


In [5]:
# class_dict.csv -> label_map
import pandas as pd
import numpy as np

# Učitaj class_dict i kreiraj mapu: (R, G, B) → label_index
df = pd.read_csv("data/class_dict.csv")
COLOR2LABEL = {tuple([row.r, row.g, row.b]): idx for idx, row in df.iterrows()}


Dataset klasa(PyTorch)


In [6]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

class DeepGlobeDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def rgb_to_label(self, mask):
        mask_array = np.array(mask)
        label_mask = np.zeros(mask_array.shape[:2], dtype=np.uint8)

        for rgb, label in COLOR2LABEL.items():
            matches = np.all(mask_array == rgb, axis=-1)
            label_mask[matches] = label

        return label_mask

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("RGB")

        label_mask = self.rgb_to_label(mask)

        if self.transform:
            augmented = self.transform(image=np.array(image), mask=label_mask)
            image = augmented["image"]
            label_mask = augmented["mask"]

        return image, label_mask.long()


Transformacije


In [7]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(),
    ToTensorV2()
])


Priprema putanja za tren i val


In [25]:
import os
from sklearn.model_selection import train_test_split

image_dir = "data/train"

# Pronađi sve satelitske slike za koje postoji odgovarajuća .png maska
image_files = sorted([
    f for f in os.listdir(image_dir)
    if f.endswith("_sat.jpg") and os.path.exists(os.path.join(image_dir, f.replace("_sat.jpg", "_mask.png")))
])

# Napravi liste punih putanja
image_paths = [os.path.join(image_dir, f) for f in image_files]
mask_paths = [os.path.join(image_dir, f.replace("_sat.jpg", "_mask.png")) for f in image_files]

# Podjela: 70% train, 15% val, 15% test
train_imgs, valtest_imgs, train_masks, valtest_masks = train_test_split(
    image_paths, mask_paths, test_size=0.30, random_state=42)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    valtest_imgs, valtest_masks, test_size=0.5, random_state=42)

print(f"✅ Train: {len(train_imgs)}, Val: {len(val_imgs)}, Test: {len(test_imgs)}")


✅ Train: 562, Val: 120, Test: 121


Dataset i DataLoader


In [26]:
from torch.utils.data import DataLoader

# Transformacije – koristiš iste za validaciju i test, osim augmentacija
train_dataset = DeepGlobeDataset(train_imgs, train_masks, transform=train_transform)
val_dataset = DeepGlobeDataset(val_imgs, val_masks, transform=train_transform)  # bez augmentacije u eval!
test_dataset = DeepGlobeDataset(test_imgs, test_masks, transform=train_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)


**U NET**

In [11]:
!pip install segmentation-models-pytorch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [12]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",        # Backbone
    encoder_weights="imagenet",     # Pretrained
    in_channels=3,                  # RGB
    classes=7,                      # 7 klasa u DeepGlobeu
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Loss,optimitzer i metric

In [13]:
import torch
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [14]:
import numpy as np

def compute_iou(preds, labels, num_classes):
    ious = []
    preds = preds.view(-1)
    labels = labels.view(-1)

    for cls in range(num_classes):
        pred_inds = preds == cls
        label_inds = labels == cls
        intersection = (pred_inds & label_inds).sum().item()
        union = (pred_inds | label_inds).sum().item()
        if union == 0:
            ious.append(float('nan'))  # klasa nije prisutna
        else:
            ious.append(intersection / union)
    return np.nanmean(ious), ious


***TRENING I VALIDACIJE***

In [15]:
def train_loop(model, train_loader, val_loader, optimizer, loss_fn, device, epochs=10, save_path="best_model.pth"):
    model.to(device)
    best_val_loss = float("inf")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    for epoch in range(1, epochs + 1):
        print(f"\n📦 Epoch {epoch}/{epochs}")

        model.train()
        train_loss = 0
        correct_pixels = 0
        total_pixels = 0

        for images, masks in tqdm(train_loader):
            images, masks = images.to(device), masks.to(device)

            preds = model(images)
            loss = loss_fn(preds, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            pred_classes = torch.argmax(preds, dim=1)
            correct_pixels += (pred_classes == masks).sum().item()
            total_pixels += torch.numel(masks)

        train_loss /= len(train_loader)
        train_acc = correct_pixels / total_pixels

        # VALIDACIJA
        model.eval()
        val_loss = 0
        correct_pixels = 0
        total_pixels = 0
        all_preds = []
        all_masks = []

        with torch.no_grad():
            for images, masks in tqdm(val_loader):
                images, masks = images.to(device), masks.to(device)

                preds = model(images)
                loss = loss_fn(preds, masks)
                val_loss += loss.item()

                pred_classes = torch.argmax(preds, dim=1)
                correct_pixels += (pred_classes == masks).sum().item()
                total_pixels += torch.numel(masks)

                all_preds.append(pred_classes.cpu())
                all_masks.append(masks.cpu())

        val_loss /= len(val_loader)
        val_acc = correct_pixels / total_pixels

        # Spoji sve predikcije za IoU
        all_preds = torch.cat(all_preds, dim=0)
        all_masks = torch.cat(all_masks, dim=0)
        mean_iou, iou_per_class = compute_iou(all_preds, all_masks, num_classes=7)

        print(f"🔹 Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")
        print(f"🎯 Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")
        print(f"📊 Val mean IoU: {mean_iou:.4f}")
        print(f"📈 IoU per class: {['{:.2f}'.format(i) for i in iou_per_class]}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Novi najbolji model spremljen u '{save_path}'")


In [20]:
import torch

if torch.cuda.is_available():
    print("✅ GPU aktivan:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU NIJE aktivan – koristiš CPU.")


✅ GPU aktivan: Tesla T4


In [21]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Tesla T4


Glavna train loop fja


In [18]:
from tqdm import tqdm
import numpy as np
import torch

def compute_iou(preds, labels, num_classes):
    ious = []
    preds = preds.view(-1)
    labels = labels.view(-1)

    for cls in range(num_classes):
        pred_inds = preds == cls
        label_inds = labels == cls
        intersection = (pred_inds & label_inds).sum().item()
        union = (pred_inds | label_inds).sum().item()
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)
    return np.nanmean(ious), ious


def train_loop(model, train_loader, val_loader, optimizer, loss_fn, device, epochs=10, save_path="best_model.pth", num_classes=7):
    model.to(device)
    best_val_loss = float("inf")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    for epoch in range(1, epochs + 1):
        print(f"\n📦 Epoch {epoch}/{epochs}")

        model.train()
        train_loss = 0
        correct_pixels = 0
        total_pixels = 0

        for images, masks in tqdm(train_loader):
            images, masks = images.to(device), masks.to(device)

            preds = model(images)
            loss = loss_fn(preds, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            pred_classes = torch.argmax(preds, dim=1)
            correct_pixels += (pred_classes == masks).sum().item()
            total_pixels += torch.numel(masks)

        train_loss /= len(train_loader)
        train_acc = correct_pixels / total_pixels

        # VALIDACIJA
        model.eval()
        val_loss = 0
        correct_pixels = 0
        total_pixels = 0
        all_preds = []
        all_masks = []

        with torch.no_grad():
            for images, masks in tqdm(val_loader):
                images, masks = images.to(device), masks.to(device)

                preds = model(images)
                loss = loss_fn(preds, masks)
                val_loss += loss.item()

                pred_classes = torch.argmax(preds, dim=1)
                correct_pixels += (pred_classes == masks).sum().item()
                total_pixels += torch.numel(masks)

                all_preds.append(pred_classes.cpu())
                all_masks.append(masks.cpu())

        val_loss /= len(val_loader)
        val_acc = correct_pixels / total_pixels

        # Spoji sve predikcije za IoU
        all_preds = torch.cat(all_preds, dim=0)
        all_masks = torch.cat(all_masks, dim=0)
        mean_iou, iou_per_class = compute_iou(all_preds, all_masks, num_classes)

        print(f"🔹 Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")
        print(f"🎯 Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")
        print(f"📊 Val mean IoU: {mean_iou:.4f}")
        print(f"📈 IoU per class: {[f'{iou:.2f}' if not np.isnan(iou) else 'NaN' for iou in iou_per_class]}")

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Novi najbolji model spremljen u '{save_path}'")


POKRETANJE TRENINGA

In [ ]:
# 📌 Definiraj uređaj
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ▶️ Pokreni trening petlju
train_loop(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=10,
    save_path="best_model.pth",
    num_classes=7
)



📦 Epoch 1/10


  0%|          | 0/71 [00:00<?, ?it/s]

**NAJBOLJI MODEL EVALUACIJA I REZULTATI**

In [ ]:
# Učitaj najbolji model nakon treniranja
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()


Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [ ]:
import pandas as pd

val_df = pd.DataFrame({
    "image_path": val_imgs,
    "mask_path": val_masks
})


In [ ]:
val_dataset = LandCoverDataset(
    val_df,
    augmentation=get_validation_augmentation(),  # već definirana ranije
)

val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)


Spremljeno: train.csv


ValueError: All arrays must be of the same length

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=device))
model.to(device)
model.eval()


In [ ]:
class_names = ['Urban', 'Agriculture', 'Rangeland', 'Forest', 'Water', 'Barren', 'Unknown']

mean_ious, accs_per_class, conf_matrix = full_evaluation(
    model, val_loader, device, num_classes=7, class_names=class_names
)


Evaluacija

In [ ]:
def compute_iou_per_class(pred, target, num_classes):
    ious = []
    for cls in range(num_classes):
        pred_inds = (pred == cls)
        target_inds = (target == cls)
        intersection = (pred_inds & target_inds).sum()
        union = (pred_inds | target_inds).sum()
        if union == 0:
            ious.append(np.nan)  # klasa se ne pojavljuje
        else:
            ious.append(intersection / union)
    return ious


In [ ]:
import sklearn.metrics as skm

def compute_confusion_matrix(y_true, y_pred, num_classes):
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    return skm.confusion_matrix(y_true_flat, y_pred_flat, labels=list(range(num_classes)))


In [ ]:
def compute_accuracy_per_class(conf_matrix):
    accs = []
    for i in range(conf_matrix.shape[0]):
        tp = conf_matrix[i, i]
        total = conf_matrix[i, :].sum()
        accs.append(tp / total if total > 0 else np.nan)
    return accs


In [ ]:
from tqdm import tqdm

def full_evaluation(model, loader, device, num_classes=7, class_names=None):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for images, masks in tqdm(loader):
            images = images.to(device)
            preds = model(images)
            preds = torch.argmax(preds, dim=1).cpu().numpy()
            masks = masks.cpu().numpy()

            all_preds.append(preds)
            all_targets.append(masks)

    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # mIoU
    all_ious = []
    for p, t in zip(all_preds, all_targets):
        ious = compute_iou_per_class(p, t, num_classes)
        all_ious.append(ious)
    mean_ious = np.nanmean(np.array(all_ious), axis=0)
    mean_iou_total = np.nanmean(mean_ious)

    # Confusion matrix
    conf_matrix = compute_confusion_matrix(all_targets, all_preds, num_classes)
    accs_per_class = compute_accuracy_per_class(conf_matrix)

    # Ispis rezultata
    print("\n📊 Evaluacija po klasama:")
    for i, iou in enumerate(mean_ious):
        name = class_names[i] if class_names else f"Class {i}"
        acc = accs_per_class[i]
        print(f"{name}: IoU={iou:.4f}, Acc={acc:.4f}")

    print(f"\n🔹 Mean IoU: {mean_iou_total:.4f}")
    print(f"🔹 Overall pixel accuracy: {(all_preds == all_targets).sum() / all_targets.size:.4f}")

    return mean_ious, accs_per_class, conf_matrix


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_conf_matrix(conf_matrix, class_names):
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()


100%|██████████| 172/172 [00:15<00:00, 11.05it/s]

🌿 Postotak zelenih površina na test skupu: 81.87%


np.float64(81.87057140261628)

In [ ]:
class_names = ['Urban', 'Agriculture', 'Rangeland', 'Forest', 'Water', 'Barren', 'Unknown']
mean_ious, accs_per_class, conf_matrix = full_evaluation(model, val_loader, device, num_classes=7, class_names=class_names)

plot_conf_matrix(conf_matrix, class_names)
